### **Introduction to Subspace Clustering**

**Overview**
This notebook explores Subspace Clustering, a specialized branch of unsupervised learning designed for high-dimensional data. In many real-world scenarios, clusters are hidden because they only exist in a subset of features; in the full space, "noise" from irrelevant dimensions makes traditional tools like K-Means fail.

**The Two Approaches**
We are comparing two mathematically distinct philosophies:

1. FINDIT (Top-Down): A heuristic approach. It starts with all dimensions and uses
a "Dimension Voting" strategy to discard irrelevant ones for each cluster.

2. SSC (Bottom-Up): An optimization approach. It uses Sparse Representation to find which points belong in the same linear subspace.

**Why the Breast Cancer Dataset?**
The Wisconsin Diagnostic Breast Cancer (WDBC) dataset is ideal because:

1. High Dimensionality: It has 30 features, creating a "curse of dimensionality" risk.

2. Feature Redundancy: Features like "radius," "perimeter," and "area" are highly correlated. Subspace algorithms help identify which specific measurements truly define a tumor's malignancy.

3. We have actual labels (Malignant/Benign) to validate if our unsupervised algorithms actually discovered the correct biological groups.

In [ ]:
# Importing pandas for handling data
import pandas as pd

# Loading dataset from the git hub link
url = "https://raw.githubusercontent.com/hduperera/Datasets/refs/heads/main/glass.csv"
df = pd.read_csv(url)

# Displaying first few rows to understanding the dataset
df.head()

,Id,RI,Na,Mg,Al,Si,K,Ca,Ba,Fe,Type of glass
0,1,1.52101,13.64,4.49,1.10,71.78,0.06,8.75,0.0,0.0,1
1,2,1.51761,13.89,3.60,1.36,72.73,0.48,7.83,0.0,0.0,1
2,3,1.51618,13.53,3.55,1.54,72.99,0.39,7.78,0.0,0.0,1
3,4,1.51766,13.21,3.69,1.29,72.61,0.57,8.22,0.0,0.0,1
4,5,1.51742,13.27,3.62,1.24,73.08,0.55,8.07,0.0,0.0,1


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

## Data Preparation
Before clustering, we must:

> Remove non-numeric columns (ID and empty columns).

> Scale the data: Both algorithms rely on distances. Without scaling, a feature with large numbers (like Area) would overpower features with small numbers (like Smoothness).

In [ ]:
#Importing necessary libaries
from sklearn.preprocessing import MinMaxScaler #For sclaing
import numpy as np

#Cleaning
X = df.drop(['id', 'diagnosis', 'Unnamed: 32'], axis=1)
y = df['diagnosis'].map({'M': 1, 'B': 0})

#Scaling
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

print(f"Dataset loaded with {X_scaled.shape[1]} features and {X_scaled.shape[0]} samples.")

Dataset loaded with 30 features and 569 samples.


## The Top-Down Strategy (FINDIT)
FINDIT works like a committee. It picks a "Medoid" (a center point) and asks its neighbors: "In which dimensions are we all actually similar?" If 80% of neighbors agree a dimension is important, it is kept for that cluster.

In [ ]:
def findit_logic(data, n_clusters=2, epsilon=0.1, tau=0.8):
    # 1. Choose Medoids
    indices = np.random.choice(data.shape[0], n_clusters, replace=False)
    medoids = data[indices]

    # 2. Voting for relevant dimensions
    subspaces = []
    for m in medoids:
        dists = np.linalg.norm(data - m, axis=1)
        neighbors = data[np.argsort(dists)[1:15]] # Top 14 neighbors
        votes = np.sum(np.abs(neighbors - m) <= epsilon, axis=0) / 14
        subspaces.append(np.where(votes >= tau)[0])

    # 3. Assign points to the best subspace
    labels = []
    for i in range(data.shape[0]):
        dists = [np.linalg.norm(data[i, s] - medoids[idx, s]) if len(s) > 0 else 999
                 for idx, s in enumerate(subspaces)]
        labels.append(np.argmin(dists))
    return np.array(labels), subspaces

findit_labels, findit_subspaces = findit_logic(X_scaled)

## The Bottom-Up Strategy (Sparse Subspace Clustering)
SSC is more advanced. It assumes that a Malignant sample can be perfectly "reconstructed" by using a small weighted sum of other Malignant samples. It builds a map of these connections to see who belongs together.

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.cluster import SpectralClustering

def ssc_logic(data, alpha=0.01):
    n = data.shape[0]
    C = np.zeros((n, n))
    for i in range(n):
        lasso = Lasso(alpha=alpha, fit_intercept=False)
        Xi = np.delete(data, i, axis=0)
        lasso.fit(Xi.T, data[i])
        C[i, :i], C[i, i+1:] = lasso.coef_[:i], lasso.coef_[i:]

    W = np.abs(C) + np.abs(C).T
    sc = SpectralClustering(n_clusters=2, affinity='precomputed')
    return sc.fit_predict(W)

ssc_labels = ssc_logic(X_scaled)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.272e-03, tolerance: 8.178e-04
  model = cd_fast.enet_coordinate_descent(


How did they do?
We use the Adjusted Rand Index (ARI).

Score of 1.0: The algorithm perfectly matched the doctors' diagnosis.

Score of 0.0: The algorithm just guessed randomly.

In [ ]:
from sklearn.metrics import adjusted_rand_score

print(f"FINDIT Accuracy (ARI): {adjusted_rand_score(y, findit_labels):.4f}")
print(f"SSC Accuracy (ARI):    {adjusted_rand_score(y, ssc_labels):.4f}")

FINDIT Accuracy (ARI): 0.1908
SSC Accuracy (ARI):    0.1931


In [ ]:
def create_adaptive_grid(df, alpha=0.5, threshold_bins=10):
    """
    Creates adaptive intervals for each dimension.
    alpha: density threshold factor
    """
    grid = {}
    for col in df.columns:
        # Initial fine-grained histogram
        hist, bin_edges = np.histogram(df[col], bins=threshold_bins)
        avg_density = np.mean(hist)

        # Merge bins that are similar in density (Adaptive Logic)
        intervals = []
        for i in range(len(hist)):
            if hist[i] >= alpha * avg_density:
                intervals.append((bin_edges[i], bin_edges[i+1]))
        grid[col] = intervals
    return grid

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_mafia_subspace(df, dim1, dim2, grid):
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(df[dim1], df[dim2], alpha=0.3, s=10, color='gray')

    # Draw the adaptive grid for these two dimensions
    for interval_x in grid[dim1]:
        for interval_y in grid[dim2]:
            rect = patches.Rectangle((interval_x[0], interval_y[0]),
                                     interval_x[1]-interval_x[0],
                                     interval_y[1]-interval_y[0],
                                     linewidth=1, edgecolor='r', facecolor='none')
            ax.add_patch(rect)

    ax.set_xlabel(dim1)
    ax.set_ylabel(dim2)
    ax.set_title(f"MAFIA Adaptive Grid: {dim1} vs {dim2}")
    plt.show()

# Example usage with two breast cancer features
plot_mafia_subspace(X_scaled, 'mean radius', 'mean texture', grid)

NameError: name 'X_scaled' is not defined